# Beam Search Evaluation for Dialect-Conditioned ASR

This notebook performs beam search decoding on the validation set and computes WER metrics.

**Important**: This uses the same random seed (17) as training to ensure identical train/val split.

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchmetrics.text import WordErrorRate
from transformers import WhisperProcessor

from model import DialectConditionedWhisper
from dataset import get_validation_loader

## Configuration

In [ ]:
# Paths
BASE_PATH = "E:/Work/My Papers/Dravidian Lang Tech 2026/Dialect based speech processing"
CSV_PATH = f"{BASE_PATH}/Final Dataset/Train/transcripts.csv"
EMBEDDINGS_PATH = f"{BASE_PATH}/Automatic Speech Recognition/dialect_embeddings.npz"
CHECKPOINT_PATH = f"{BASE_PATH}/Automatic Speech Recognition/train_data/checkpoints/training 1/best_model.pth"
RESULTS_BASE_PATH = f"{BASE_PATH}/Automatic Speech Recognition/train_data/beamsearch_results"

# Model
WHISPER_MODEL = "vasista22/whisper-tamil-medium"
DIALECT_EMBEDDING_DIM = 768

# Beam search parameters
NUM_BEAMS = 5
MAX_LENGTH = 448  # Max tokens to generate
BATCH_SIZE = 2  # Smaller batch for beam search

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## Create Result Folder

Automatically creates `result_1`, `result_2`, etc.

In [ ]:
def get_next_result_folder(base_path):
    """Get the next available result folder (result_1, result_2, etc.)"""
    os.makedirs(base_path, exist_ok=True)
    
    existing = [d for d in os.listdir(base_path) if d.startswith('result_')]
    if not existing:
        next_num = 1
    else:
        nums = [int(d.split('_')[1]) for d in existing if d.split('_')[1].isdigit()]
        next_num = max(nums) + 1 if nums else 1
    
    result_folder = os.path.join(base_path, f'result_{next_num}')
    os.makedirs(result_folder, exist_ok=True)
    
    return result_folder

RESULT_FOLDER = get_next_result_folder(RESULTS_BASE_PATH)
print(f"Saving results to: {RESULT_FOLDER}")

## Load Model and Data

In [ ]:
# Load processor
print("Loading Whisper processor...")
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL, task="transcribe")

# Load model
print("Loading dialect-conditioned Whisper model...")
model = DialectConditionedWhisper(
    whisper_model_name=WHISPER_MODEL,
    dialect_embedding_dim=DIALECT_EMBEDDING_DIM
)

# Load checkpoint
print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Loaded model from epoch {checkpoint.get('epoch', 'unknown')}")
print(f"Best validation WER: {checkpoint.get('best_metric', 'unknown'):.2f}%")

In [ ]:
# Load validation data
print("Loading validation data...")
val_loader = get_validation_loader(
    csv_path=CSV_PATH,
    embeddings_path=EMBEDDINGS_PATH,
    processor=processor,
    batch_size=BATCH_SIZE,
    train_size=0.8,
    num_workers=2,
)

print(f"Validation batches: {len(val_loader)}")

## Beam Search Evaluation

In [ ]:
# WER metric
wer_metric = WordErrorRate()

# Storage for results
all_predictions = []
all_references = []
all_audio_ids = []
all_wers = []

print(f"Running beam search evaluation (num_beams={NUM_BEAMS})...")
print("=" * 60)

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Beam Search Decoding"):
        input_features = batch['input_features'].to(DEVICE)
        dialect_embedding = batch['dialect_embedding'].to(DEVICE)
        references = batch['transcription_text']
        audio_ids = batch['audio_ids']
        
        # Beam search generation
        generated_ids = model.generate(
            input_features=input_features,
            dialect_embedding=dialect_embedding,
            num_beams=NUM_BEAMS,
            max_length=MAX_LENGTH,
            early_stopping=True,
        )
        
        # Decode predictions
        predictions = processor.tokenizer.batch_decode(
            generated_ids, 
            skip_special_tokens=True
        )
        
        # Compute WER for each sample
        for pred, ref, aid in zip(predictions, references, audio_ids):
            sample_wer = wer_metric([pred], [ref]).item() * 100
            all_predictions.append(pred)
            all_references.append(ref)
            all_audio_ids.append(aid)
            all_wers.append(sample_wer)

print("\nEvaluation complete!")

## Compute Metrics

In [ ]:
# Overall WER
overall_wer = wer_metric(all_predictions, all_references).item() * 100
mean_wer = np.mean(all_wers)
std_wer = np.std(all_wers)
median_wer = np.median(all_wers)

print("\n" + "=" * 60)
print("BEAM SEARCH EVALUATION RESULTS")
print("=" * 60)
print(f"Number of samples:    {len(all_predictions)}")
print(f"Beam width:           {NUM_BEAMS}")
print(f"\nOverall WER:          {overall_wer:.2f}%")
print(f"Mean WER:             {mean_wer:.2f}%")
print(f"Std WER:              {std_wer:.2f}%")
print(f"Median WER:           {median_wer:.2f}%")
print(f"Min WER:              {min(all_wers):.2f}%")
print(f"Max WER:              {max(all_wers):.2f}%")
print("=" * 60)

## Save Results

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'audio_id': all_audio_ids,
    'reference': all_references,
    'prediction': all_predictions,
    'wer': all_wers
})

# Sort by WER (worst first for analysis)
results_df_sorted = results_df.sort_values('wer', ascending=False)

# Save to CSV
csv_path = os.path.join(RESULT_FOLDER, 'predictions.csv')
results_df_sorted.to_csv(csv_path, index=False)
print(f"Saved predictions to: {csv_path}")

# Save metrics summary
metrics = {
    'timestamp': datetime.now().isoformat(),
    'checkpoint_path': CHECKPOINT_PATH,
    'num_samples': len(all_predictions),
    'num_beams': NUM_BEAMS,
    'overall_wer': overall_wer,
    'mean_wer': mean_wer,
    'std_wer': std_wer,
    'median_wer': median_wer,
    'min_wer': min(all_wers),
    'max_wer': max(all_wers),
    'greedy_wer_from_training': checkpoint.get('best_metric', None)
}

metrics_path = os.path.join(RESULT_FOLDER, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to: {metrics_path}")

## Visualizations

In [ ]:
# WER Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(all_wers, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(overall_wer, color='red', linestyle='--', linewidth=2, label=f'Overall WER: {overall_wer:.2f}%')
axes[0].axvline(median_wer, color='orange', linestyle='--', linewidth=2, label=f'Median WER: {median_wer:.2f}%')
axes[0].set_xlabel('WER (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('WER Distribution (Beam Search)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(all_wers, vert=True)
axes[1].set_ylabel('WER (%)')
axes[1].set_title('WER Box Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULT_FOLDER, 'wer_distribution.png')
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved plot to: {fig_path}")

In [ ]:
# Compare Greedy vs Beam Search
greedy_wer = checkpoint.get('best_metric', None)

if greedy_wer:
    fig, ax = plt.subplots(figsize=(8, 6))
    
    methods = ['Greedy Decoding', f'Beam Search (n={NUM_BEAMS})']
    wers = [greedy_wer, overall_wer]
    colors = ['#3498db', '#27ae60']
    
    bars = ax.bar(methods, wers, color=colors, edgecolor='white', linewidth=2)
    
    # Add value labels
    for bar, wer in zip(bars, wers):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{wer:.2f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax.set_ylabel('WER (%)', fontsize=12)
    ax.set_title('Decoding Method Comparison', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Calculate improvement
    improvement = greedy_wer - overall_wer
    improvement_pct = (improvement / greedy_wer) * 100
    
    ax.text(0.5, 0.95, f'Improvement: {improvement:.2f}% ({improvement_pct:.1f}% relative)',
            transform=ax.transAxes, ha='center', fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    fig_path = os.path.join(RESULT_FOLDER, 'greedy_vs_beam.png')
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved plot to: {fig_path}")

## Sample Predictions

In [ ]:
# Show best and worst predictions
print("\n" + "=" * 60)
print("BEST PREDICTIONS (Lowest WER)")
print("=" * 60)

for i, row in results_df_sorted.tail(5).iterrows():
    print(f"\nAudio ID: {row['audio_id']}")
    print(f"WER: {row['wer']:.2f}%")
    print(f"Reference:  {row['reference'][:100]}..." if len(row['reference']) > 100 else f"Reference:  {row['reference']}")
    print(f"Prediction: {row['prediction'][:100]}..." if len(row['prediction']) > 100 else f"Prediction: {row['prediction']}")

In [ ]:
print("\n" + "=" * 60)
print("WORST PREDICTIONS (Highest WER)")
print("=" * 60)

for i, row in results_df_sorted.head(5).iterrows():
    print(f"\nAudio ID: {row['audio_id']}")
    print(f"WER: {row['wer']:.2f}%")
    print(f"Reference:  {row['reference'][:100]}..." if len(row['reference']) > 100 else f"Reference:  {row['reference']}")
    print(f"Prediction: {row['prediction'][:100]}..." if len(row['prediction']) > 100 else f"Prediction: {row['prediction']}")

In [ ]:
print("\n" + "=" * 60)
print(f"\nAll results saved to: {RESULT_FOLDER}")
print("Files created:")
for f in os.listdir(RESULT_FOLDER):
    print(f"  - {f}")